In [1]:
# Imports
import os
import datetime

import pandas as pd
import numpy as np
from pyproj import CRS, Transformer

In [2]:
def ll2xy(lon, lat):
    """
    Transform coordinates from input geodetic coordinates (lon, lat)
    to output Antarctic Polar Stereographic coordinates (x, y)

    Parameters
    lon - Geodetic longitude in EPSG:4326 [float]
    lat - Geodetic latitude in EPSG:4326 [float]

    Returns
    x - Antarctic Polar Stereographic (EPSG:3031) x [float]
    y - Antarctic Polar Stereographic (EPSG:3031) y [float]
    """

    crs_ll = CRS("EPSG:4326")
    crs_xy = CRS("EPSG:3031")
    ll_to_xy = Transformer.from_crs(crs_ll, crs_xy, always_xy=True)
    x, y = ll_to_xy.transform(lon, lat)
    return x, y

In [3]:
def load(file):
    """
    Load processed gps data file into pandas table.

    Parameters
    file - .pos precise point solution file, Natural Resources Canada [string]

    Returns
    data - Pandas DataFrame with the following columns, extracted from file
        time - Time as datetime object
        day_of_year - Time as julian day
    flip - flag to flip order of readings for 2024 csrs-ppp data
    """

    data = pd.DataFrame()  # Create Pandas DataFrame
    flip = False
    # Read data file

    # Convert longitude and latitude from deg min sec to fractional degrees
    # Three different file formats so try one first and try the other if it
    # throws a not found exception.

    try:
        # CSRS-PPP 2024
        d = pd.read_csv(file, skiprows=3, sep="\s+")
        data["longitude"] = d["LONDD"] - d["LONMN"] / 60 - d["LONSS"] / 60 / 60
        data["latitude"] = d["LATDD"] - d["LATMN"] / 60 - d["LATSS"] / 60 / 60
        data["time"] = pd.to_datetime(d["YEAR-MM-DD"] + "T" + d["HR:MN:SS.SS"])
        data["day_of_year"] = d["DAYofYEAR"]

        # Look at data and decide if to flip
        if len(data.index) > 1:
            diff = data["time"].iloc[0] - data["time"].iloc[1]
            if diff > datetime.timedelta(seconds=0):
                flip = True
    except:  # noqa: E722
        try:
            d = pd.read_csv(file, skiprows=7, delim_whitespace=True)
            data["longitude"] = d["LON(d)"] - d["LON(m)"] / 60 - d["LON(s)"] / 60 / 60
            data["latitude"] = d["LAT(d)"] - d["LAT(m)"] / 60 - d["LAT(s)"] / 60 / 60
            data["time"] = pd.to_datetime(d["YEAR-MM-DD"] + "T" + d["HR:MN:SS.SSS"])
            data["day_of_year"] = d["DOY"]
        except:  # noqa: E722
            try:
                d = pd.read_csv(file, skiprows=5, delim_whitespace=True)
                data["longitude"] = d["LONDD"] - d["LONMN"] / 60 - d["LONSS"] / 60 / 60
                data["latitude"] = d["LATDD"] - d["LATMN"] / 60 - d["LATSS"] / 60 / 60
                data["time"] = pd.to_datetime(d["YEAR-MM-DD"] + "T" + d["HR:MN:SS.SS"])
                data["day_of_year"] = d["DAYofYEAR"]
            except:  # noqa: E722
                d = pd.read_csv(file, skiprows=6, delim_whitespace=True)
                data["longitude"] = (
                    d["LON(d)"] - d["LON(m)"] / 60 - d["LON(s)"] / 60 / 60
                )
                data["latitude"] = (
                    d["LAT(d)"] - d["LAT(m)"] / 60 - d["LAT(s)"] / 60 / 60
                )
                data["time"] = pd.to_datetime(d["YEAR-MM-DD"] + "T" + d["HR:MN:SS.SSS"])
                data["day_of_year"] = d["DOY"]

    x, y = ll2xy(lon=data["longitude"], lat=data["latitude"])
    data["x"] = x
    data["y"] = y

    x0 = data["x"][0]
    y0 = data["y"][0]

    data["dist"] = np.sqrt((data["x"] - x0) ** 2 + (data["y"] - y0) ** 2)
    return data, flip

In [4]:
def datastream(dir, years):
    """
    Takes input directory dir with year subdirectories containign .pos files.
    Starting from the first file in the first year, append days until all files
    in the directory have been added.

    Parameters
    dir - Input directory tree with structure described in top comment [string]
    years - years to be run [arr of strings]

    Returns
    data - Resulting multiday time series, data. [DataFrame]
    """
    data = pd.DataFrame()
    for year in years:
        for folder in os.scandir(dir.path):
            if folder.is_dir():  # folder is an os.DirEntry object
                if folder.name == year:
                    print(folder.name)
                    for gps in os.listdir(folder.path):
                        if gps.endswith(".pos") and not gps.startswith(
                            "."
                        ):  # Ignore xyzt, zip files
                            ind_data, flip = load(folder.path + "/" + gps)
                            if flip:  # Reorder the data if using CSRS-PPP 2024 pre 2018
                                ind_data = ind_data.reindex(index=ind_data.index[::-1])
                            data = pd.concat([data, ind_data], ignore_index=True)
    return data

In [7]:
dir = "/mnt/e/csrs_2024/all"
custom_order = [
    "la01",
    "la02",
    "la03",
    "la04",
    "la05",
    "la06",
    "la07",
    "la08",
    "la09",
    "la10",
    "la11",
    "la12",
    "la13",
    "la14",
    "la15",
    "la16",
    "la17",
    "la18",
    "slw1",
    "ws04",
    "ws05",
    "gz01",
    "gz02",
    "gz03",
    "gz04",
    "gz05",
    "gz06",
    "gz07",
    "gz08",
    "gz09",
    "gz10",
    "gz11",
    "gz12",
    "gz13",
    "gz14",
    "gz15",
    "gz16",
    "gz17",
    "gz18",
    "gz19",
    "gz20",
    "mg01",
    "mg02",
    "mg03",
    "mg04",
    "mg05",
    "mg06",
    "mg07",
]

custom_order.reverse()
# Build a dictionary from os.scandir
entries = {entry.name: entry for entry in os.scandir(dir)}

ordered_dirs = []
# Iterate in your specified order
for name in custom_order:
    if name in entries:
        sta = entries[name]
        ordered_dirs.append(sta)

In [27]:
first_and_last = {"sta": [], "first_x": [], "first_y": [], "last_x": [], "last_y": []}

for sta in ordered_dirs[:]:
    if sta.is_dir():
        print(sta.name, sta.path)

        # Get first file from first year and last file from last year
        entries = {entry.name: entry for entry in os.scandir(sta)}
        year_dirs = sorted(entries.keys())
        first_year = year_dirs[0]
        last_year = year_dirs[-1]

        print(f"First year: {first_year}, Last year: {last_year}")
        # Load first file from first year
        first_year_path = os.path.join(sta.path, first_year)
        first_file = sorted(os.listdir(first_year_path))[0]  # Get the first file
        first_file_path = os.path.join(first_year_path, first_file)
        data_first, flip = load(first_file_path)

        # Load last file from last year
        last_year_path = os.path.join(sta.path, last_year)
        last_file = sorted(os.listdir(last_year_path))[-1]  # Get the last
        last_file_path = os.path.join(last_year_path, last_file)
        data_last, flip = load(last_file_path)

        first_x = data_first["x"].iloc[0]
        first_y = data_first["y"].iloc[0]
        last_x = data_last["x"].iloc[-1]
        last_y = data_last["y"].iloc[-1]

        first_and_last["first_x"].append(first_x)
        first_and_last["first_y"].append(first_y)
        first_and_last["last_x"].append(last_x)
        first_and_last["last_y"].append(last_y)
        first_and_last["sta"].append(sta.name)

mg07 /mnt/e/csrs_2024/all/mg07
First year: 2014, Last year: 2014
mg06 /mnt/e/csrs_2024/all/mg06
First year: 2014, Last year: 2014
mg05 /mnt/e/csrs_2024/all/mg05
First year: 2014, Last year: 2014
mg04 /mnt/e/csrs_2024/all/mg04
First year: 2014, Last year: 2016
mg03 /mnt/e/csrs_2024/all/mg03
First year: 2014, Last year: 2014
mg02 /mnt/e/csrs_2024/all/mg02
First year: 2014, Last year: 2014
mg01 /mnt/e/csrs_2024/all/mg01
First year: 2014, Last year: 2014
gz20 /mnt/e/csrs_2024/all/gz20
First year: 2016, Last year: 2019
gz19 /mnt/e/csrs_2024/all/gz19
First year: 2014, Last year: 2016
gz18 /mnt/e/csrs_2024/all/gz18
First year: 2012, Last year: 2014
gz17 /mnt/e/csrs_2024/all/gz17
First year: 2012, Last year: 2014
gz16 /mnt/e/csrs_2024/all/gz16
First year: 2011, Last year: 2014
gz15 /mnt/e/csrs_2024/all/gz15
First year: 2011, Last year: 2014
gz14 /mnt/e/csrs_2024/all/gz14
First year: 2012, Last year: 2014
gz13 /mnt/e/csrs_2024/all/gz13
First year: 2012, Last year: 2014
gz12 /mnt/e/csrs_2024/all

In [29]:
df = pd.DataFrame(first_and_last)
df = df.round(2)
df.to_csv("sta_positions.csv", index=False)